In [1]:
from pyuvdata import UVData
import numpy as np

In [4]:
with open(f"{working_directory}/{bits}bit/{day}/{config_file_name}", "r") as f:
    config = json.load(f)
    dir_parents = []
    coords = []
    # unpack information from the json file
    # Call get_starting_index for all antennas except reference
    for i, (ant, details) in enumerate(config["antennas"].items()):
        if (i == 0) or (i ==baseline_idx):
            coords.append(details['coordinates'])
            dir_parents.append(details["path"])
    global_start_time = config["correlation"]["start_timestamp"]
    end_t = config["correlation"]["end_timestamp"]
    v_acclen = config["correlation"]["vis_acclen"]
    visibility_window = config["correlation"]["visibility_window"]
    T_SPECTRA = config["correlation"]["point_PFB"] / config["correlation"]["sample_rate"]

ref_coords = coords[0]
v_nchunks = int((visibility_window)/(v_acclen* T_SPECTRA))

context = [visibility_window, T_SPECTRA, v_acclen, v_nchunks, ref_coords]
print(context)
print(global_start_time)

NameError: name 'working_directory' is not defined

In [3]:
# Initialize UVData object
uv = UVData()


# Define minimal parameters
Nfreqs = 18
Nbls = 1
Npols = 4
Ntimes = 20
Nblts = Ntimes * Nbls  #assume no lost data for now


#some useful parameters
uv.Nbls = Nbls
uv.Nblts = Nblts
uv.Ntimes = Ntimes
uv.Nfreqs = Nfreqs
uv.Npols = Npols
uv.Nspws = 1
uv.Nphase = 1
uv.Nants_telescope = 2


# Create fake data arrays
data_array = np.ones((Nblts, Nfreqs, Npols), dtype=np.complex64)
flag_array = np.zeros((Nbls * Ntimes, Nfreqs, Npols), dtype=bool)
nsample_array = np.ones((Nbls * Ntimes, Nfreqs, Npols), dtype=np.float32)


#define main data
uv.data_array = data_array
uv.flag_array = flag_array

#time arrays
uv.time_array = np.arange(Nblts, dtype = float)  # some Julian Date
uv.lst_array = np.arange(Nblts, dtype = float)

#number of antenna with data in them
uv.Nants_data = 2

#code to be able to tell what two antenna are present very quickly
uv.baseline_array = np.zeros(Nblts, dtype = int)

#these tell you which antenna are present for each blt
ant_1 = np.zeros(Nblts, dtype = int)
ant_1[0], ant_1[1] = 1, 1
uv.ant_1_array = ant_1
ant_2 = np.ones(Nblts, dtype = int)
ant_2[1], ant_2[2] = 0, 0
uv.ant_2_array = ant_2

#frequency stuff
uv.freq_array = np.linspace(1, 18, 18) 
uv.channel_width = np.ones((uv.Nfreqs), dtype=float)

#which polarization we are talking about. there's number conventions that correspond to certain types of pols
uv.polarization_array = np.arange(1,5, dtype = int)  # e.g., XX

#spectral window stuff
uv.spw_array = np.array([0])
uv.flex_spw_id_array = np.zeros((Nfreqs), dtype=int)

#integration time and sample sizes
uv.integration_time = np.ones(uv.Nblts)
uv.nsample_array = nsample_array

#names?
uv.telescope_name = 'FakeTelescope'
uv.antenna_names = ['ant0', 'ant1']
uv.antenna_numbers = [0,1]
uv.antenna_positions = np.zeros((2,3))
uv.instrument = 'FakeInstrument'
uv.telescope_location = (0.0, 0.0, 0.0)
uv.history = 'History'
uv.object_name = 'Object Name'

#phase center catalogue
uv.phase_center_catalog = {
    0: {
        "cat_name": "primary_center",
        "cat_type": "sidereal",
        "cat_lon": 0.0,
        "cat_lat": 0.0,
        "cat_frame": "icrs",
    }}


uv.flex_spw_id_array = np.zeros((Nfreqs), dtype=int)
uv.phase_center_app_dec = np.zeros(uv.Nblts)
uv.phase_center_app_ra = np.zeros(uv.Nblts)
uv.phase_center_frame_pa = np.zeros(uv.Nblts)
uv.phase_center_id_array = np.zeros((uv.Nblts), dtype=int)

print(uv.phase_center_catalog)
uv.uvw_array = np.ones((Nblts, 3), dtype = float)
uv.vis_units = 'uncalib'

# Write to UVH5 file
uv.write_uvh5('example_output.uvh5', clobber=True)

{0: {'cat_name': 'primary_center', 'cat_type': 'sidereal', 'cat_lon': 0.0, 'cat_lat': 0.0, 'cat_frame': 'icrs'}}


ERFA function "utcut1" yielded 20 of "dubious year (Note 3)"
ERFA function "utctai" yielded 20 of "dubious year (Note 3)"
The lst_array is not self-consistent with the time_array and telescope location. Consider recomputing with the `set_lsts_from_time_array` method.


ValueError: Some auto-correlations have non-zero uvw_array coordinates.